In [33]:
import torch
import torch.nn as nn

## The example data

In [34]:
X = torch.tensor([[3.0, 2.0],
                  [5.0, 1.0]])
y = torch.tensor([[10.0],
                  [16.0]])

The network has 2 inputs, a hidden layer with 2 neurons with ReLU, and 1 output

## Hidden layer

I set the values by hand so the math is traceable. Normally Pytorch initializes them randomly

In [35]:
hidden = nn.Linear(2, 2)        # Creates weight (2, 2) and bias (2,)

with torch.no_grad():
    hidden.weight.copy_(torch.tensor([[1.0, -2.0],  # row 0 = neuron 0
                                      [0.5, 1.0]])) # row 1 = neuron 1
    hidden.bias.copy_(torch.tensor([0.0, -1.0]))

In [36]:
output = hidden(X)
print(output)

tensor([[-1.0000,  2.5000],
        [ 3.0000,  2.5000]], grad_fn=<AddmmBackward0>)


## nn.ReLU

## Output layer: nn.Linear(2, 1)

In [37]:
output = nn.Linear(2, 1)

with torch.no_grad():
    output.weight.copy_(torch.tensor([[2.0, 3.0]]))
    output.bias.fill_(1.0)

## nn.Module: putting the layers into one model

In [38]:
class DeliveryModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(2, 2)
        self.relu   = nn.ReLU()
        self.output = nn.Linear(2, 1)

    def forward(self, x):
        x = self.hidden(x)   # (2,2) → (2,2)
        x = self.relu(x)     # (2,2) → (2,2)
        x = self.output(x)   # (2,2) → (2,1)
        return x

model = DeliveryModel()


# super().__init__() runs nn.Module’s own constructor. That creates internal dictionaries (_parameters, _modules, _buffers) where PyTorch stores everything. If you skip it, the next line self.hidden = ... raises an error because those dictionaries don’t exist yet.

# self.hidden = nn.Linear(2, 2) looks like normal attribute assignment, but nn.Module overrides __setattr__. When the value you assign is an nn.Module, it’s stored in self._modules["hidden"]. When it’s an nn.Parameter, it’s stored in self._parameters. That’s how PyTorch finds every learnable tensor later.

# forward(self, x) defines the computation. You write it, but you don’t call it directly.

# model(X) calls nn.Module.__call__, which runs any registered hooks and then calls your forward. Always use model(X), not model.forward(X).

In [39]:
for name, p in model.named_parameters():
    print(name, tuple(p.shape))

# Total 9 learnable parameters

hidden.weight (2, 2)
hidden.bias (2,)
output.weight (1, 2)
output.bias (1,)


In [40]:
with torch.no_grad():
    model.hidden.weight.copy_(torch.tensor([[1.0, -2.0], [0.5, 1.0]]))
    model.hidden.bias.copy_(torch.tensor([0.0, -1.0]))
    model.output.weight.copy_(torch.tensor([[2.0, 3.0]]))
    model.output.bias.fill_(1.0)

pred = model(X)
print(pred)   # tensor([[ 8.5], [14.5]])

tensor([[ 8.5000],
        [14.5000]], grad_fn=<AddmmBackward0>)


## Loss Function: nn.MSELoss

In [41]:
loss_function = nn.MSELoss()
loss = loss_function(pred, y)
print(loss)

tensor(2.2500, grad_fn=<MseLossBackward0>)


## Computing gradients

In [42]:
loss.backward()
print(model.output.weight.grad)  # tensor([[-4.5, -7.5]])
print(model.output.bias.grad)    # tensor([-3.])
print(model.hidden.weight.grad)  # tensor([[-15.0, -3.0], [-36.0, -13.5]])
print(model.hidden.bias.grad)    # tensor([-3., -9.])

tensor([[-4.5000, -7.5000]])
tensor([-3.])
tensor([[-15.0000,  -3.0000],
        [-36.0000, -13.5000]])
tensor([-3., -9.])


## The optimizer -> Updating parameters

In [43]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
optimizer.step()

In [45]:
for epoch in range(1000):
    pred = model(X)              # forward pass
    loss = loss_function(pred, y)      # compute scalar loss
    optimizer.zero_grad()        # reset .grad to 0
    loss.backward()              # compute new gradients
    optimizer.step()             # update parameters